# two_stage_reverse_innerOOF — Reverse Two-Stage + inner-OOF Stage 2 feature

**목적**: 기존 `two_stage_reverse_hpo` 의 best HP를 그대로 사용하되, **Stage 2 clf 학습 시 reg 예측 feature** 를 in-sample(overfit) 에서 **inner-OOF (out-of-sample)** 로 바꿔 분포 mismatch 를 제거. HPO 없이 두 모드를 같은 best HP 로 refit 후 비교.

**핵심 변경 (Stage 2 clf 학습 feature)**
```
[기존 in-sample]
reg.fit(X_tr, y_tr_log)              # outer-train 전체로 학습
reg_train = reg.predict(X_tr)        # 같은 outer-train 예측 → overfit
X_aug = [X_tr, reg_train]; clf.fit(X_aug, y_bin)
→ 학습 시 reg 예측 분포 (오차 작음) ≠ 추론 시 (오차 큼)

[inner-OOF 개선]
# 1) outer-train을 inner KFold(unit-level)로 분할
for itr, ivl in inner_kf.split(outer_tr_units):
    reg_inner.fit(X[itr]); reg_train_oof[ivl] = reg_inner.predict(X[ivl])
# 2) 추론용 reg_full 은 outer-train 전체로 별도 학습
reg_full.fit(X_tr, y_tr_log)
# 3) clf 학습 시 inner-OOF 예측(분포 일치)을 feature로
X_aug = [X_tr, reg_train_oof]; clf.fit(X_aug, y_bin)
# 4) val/test 추론은 reg_full 사용 (out-of-sample 분포)
→ 학습-추론 분포 일치
```

**inner KFold 도 반드시 unit 단위 분할** (die 단위 분할 시 같은 unit 의 4 die 가 inner-train/inner-val 에 섞여 inner leakage).

**고정 설정**
- best HP: `4_output/_temp/two_stage_reverse/best_params.json` 그대로 로드 (refit only, HPO 없음)
- 전처리: 03b log1p preset (ts-reverse 본인이 검증한 PP, val=0.005709)
- KFold 5 (outer, unit-level shuffle SEED=42)
- K_INNER=5 (정석)
- target_transform = log1p, CLIP_Y_EXTREME=True

**비교 대상**
- ts-reverse-hpo-001 (in-sample, 기존): val=0.005709, test=0.008412
- ts-reverse-innerOOF (이 노트북, in-sample mode): 동일 결과 재현 확인
- ts-reverse-innerOOF (이 노트북, innerOOF mode): val/test 변화량 측정

**격리**: `4_output/_temp/two_stage_reverse_innerOOF/{insample,innerOOF}/` 에 분리 저장. 모듈 무수정.

## 1. 환경 + import

In [1]:
import os, sys, json, time

%run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import (
    PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR,
)
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import lightgbm as lgb
from sklearn.model_selection import KFold

import logging
logging.getLogger('lightgbm').setLevel(logging.ERROR)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트


## 2. 설정 (refit only, best HP load)

기존 `ts-reverse-hpo-001` 의 best HP 를 그대로 사용. HPO 없음.

In [2]:
EXP_ID   = 'ts-reverse-innerOOF-001'
EXP_MEMO = 'Reverse Two-Stage refit (in-sample vs inner-OOF) — same best HP, no HPO'
USER     = 'jh'

N_FOLDS  = 5
K_INNER  = 5
CLIP_Y_EXTREME = True
TARGET_TRANSFORM = 'log1p'

OUT_DIR_ROOT = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_reverse_innerOOF')
OUT_DIR = {
    'insample':  os.path.join(OUT_DIR_ROOT, 'insample'),
    'innerOOF':  os.path.join(OUT_DIR_ROOT, 'innerOOF'),
}
for d in OUT_DIR.values():
    os.makedirs(d, exist_ok=True)

# ── 전처리 PARAMS (03b log1p preset = ts-reverse 본인 best) ──
PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}

# ── best HP 로드 (ts-reverse-hpo-001) ──
BEST_PARAMS_PATH = os.path.join(OUTPUT_DIR, '_temp', 'two_stage_reverse', 'best_params.json')
with open(BEST_PARAMS_PATH, 'r', encoding='utf-8') as f:
    bp = json.load(f)

best_params  = bp['best_params']
best_w0      = bp['best_w0']
best_reg_obj = bp['best_reg_obj']
best_clf_spw = bp['best_clf_spw']

# LGBM HP 만 추출 (path B 전용 인자 제거)
hp_best = {
    k: v for k, v in best_params.items()
    if k not in ['w0', 'reg_objective', 'clf_scale_pos_weight']
}
hp_best.update(random_state=SEED, n_jobs=-1, verbose=-1, subsample_freq=1)

print(f'EXP_ID            = {EXP_ID}')
print(f'N_FOLDS / K_INNER = {N_FOLDS} / {K_INNER}')
print(f'TARGET_TRANSFORM  = {TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'BEST_PARAMS_PATH  = {BEST_PARAMS_PATH}')
print(f'  best_w0         = {best_w0}')
print(f'  best_reg_obj    = {best_reg_obj}')
print(f'  best_clf_spw    = {best_clf_spw}')
print(f'OUT_DIR.insample  = {OUT_DIR["insample"]}')
print(f'OUT_DIR.innerOOF  = {OUT_DIR["innerOOF"]}')

EXP_ID            = ts-reverse-innerOOF-001
N_FOLDS / K_INNER = 5 / 5
TARGET_TRANSFORM  = log1p | CLIP_Y_EXTREME=True
BEST_PARAMS_PATH  = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse\best_params.json
  best_w0         = 0.17672657730420213
  best_reg_obj    = regression
  best_clf_spw    = 2.43
OUT_DIR.insample  = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse_innerOOF\insample
OUT_DIR.innerOOF  = c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse_innerOOF\innerOOF


## 3. 데이터 로드 + Y clip

In [3]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개 샘플')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

print(f'\n[데이터 로드] xs={xs.shape}, X feat_cols={len(feat_cols)}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  y_train: max={y_train_unit.max():.6f}, mean={y_train_unit.mean():.6f}, zero ratio={(y_train_unit==0).mean():.1%}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개 샘플

[데이터 로드] xs=(174572, 1091), X feat_cols=1087
  unit train=26,187, val=8,727, test=8,729
  y_train: max=0.097417, mean=0.002481, zero ratio=70.8%


## 4. 전처리 (03b log1p preset)

In [4]:
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float64)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float64)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float64)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)
print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  y_train_die_broadcast (unit y): mean={y_train_die_broadcast.mean():.6f}')
print(f'  y_bin_die (broadcasted y>0):    pos ratio={y_bin_die_broadcast.mean():.4f}')

# Outer KFold split (unit-level)
unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))
print(f'\nfold split: {N_FOLDS} outer folds (unit-level)')

[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation] 총 결측: 343,494
  train-only 모드: train 104,748 / 전체 174,572 행
  1단계 (공간 보간, dist<=5.0): 156,772개 채움 → 잔여: 186,722
  2단계 (lot 평균, train 기준): 105,526개 채움 → 잔여: 81,196
  3단계 (train 전체 평균): 81,196개 채움 → 잔여: 0

  [요약] 343,494 → 공간(156,772) → lot(105,526) → 전체(81,196) → 잔여(0)

[고상관 제거] threshold=0.99, keep_by=std (std)
  제거: 0개, 잔여: 564개
    [고상관 제거 2차 / imputation 후] threshold=0.99
    컬럼: 564 → 564 (0개 제거)
    DataFrame: (104748, 626)

클리닝 완

## 5. helper — `_train_path_b(mode=...)` 통합

- `mode='insample'`: 기존 방식 (reg_full 학습 → 같은 train 예측 → clf feature)
- `mode='innerOOF'`: outer-train 안에서 inner KFold 로 reg OOF 생성 → clf feature. 추론은 reg_full 사용

**inner KFold 는 반드시 unit 단위** (die 단위 분할 시 같은 unit 의 4 die 가 섞여 inner leakage).

In [5]:
def _build_reg_params(hp, reg_obj):
    p = dict(hp)
    if reg_obj.startswith('tweedie'):
        p['objective'] = 'tweedie'
        p['tweedie_variance_power'] = float(reg_obj.split('_')[1])
    else:
        p['objective'] = reg_obj
    return p


def _train_path_b(X_tr, y_tr_continuous, y_tr_bin, X_others,
                  hp, w0, reg_obj, clf_spw,
                  mode='insample', uid_tr=None, k_inner=5, seed=42):
    """Path B 1 outer fold 학습 + 예측.

    Parameters
    ----------
    mode : 'insample' or 'innerOOF'
        Stage 2 clf 학습 시 reg 예측 feature 생성 방식.
    uid_tr : np.array
        outer-train die 의 unit id (mode='innerOOF' 때 inner KFold split 용).
    X_others : list of np.array
        예측 대상 (val_split, val, test 등). 모두 reg_full 로 처리.

    Returns
    -------
    list of (prob, reg_y, final) per X in X_others.
    """
    sw_full = np.where(y_tr_continuous == 0, w0, 1.0)
    y_tr_log = np.log1p(y_tr_continuous)
    reg_params = _build_reg_params(hp, reg_obj)

    # ── Stage 1 reg_full (추론용, outer-train 전체) ──
    reg_full = lgb.LGBMRegressor(**reg_params)
    reg_full.fit(X_tr, y_tr_log, sample_weight=sw_full)

    # ── Stage 2 학습 feature: reg 예측 ──
    if mode == 'insample':
        # 같은 outer-train 에 reg_full 로 예측 (overfit, 분포 mismatch 발생)
        reg_train_log = reg_full.predict(X_tr)
        reg_train_y_for_clf = np.clip(np.expm1(reg_train_log), 0.0, None)
    elif mode == 'innerOOF':
        assert uid_tr is not None, 'innerOOF mode requires uid_tr'
        unique_inner_units = np.unique(uid_tr)
        inner_kf = KFold(n_splits=k_inner, shuffle=True, random_state=seed)
        reg_train_oof_log = np.full(len(X_tr), np.nan)
        for itr_uidx, ivl_uidx in inner_kf.split(unique_inner_units):
            itr_units = unique_inner_units[itr_uidx]
            ivl_units = unique_inner_units[ivl_uidx]
            itr_die_mask = np.isin(uid_tr, itr_units)
            ivl_die_mask = np.isin(uid_tr, ivl_units)
            sw_inner = np.where(y_tr_continuous[itr_die_mask] == 0, w0, 1.0)
            reg_inner = lgb.LGBMRegressor(**reg_params)
            reg_inner.fit(
                X_tr[itr_die_mask],
                y_tr_log[itr_die_mask],
                sample_weight=sw_inner,
            )
            reg_train_oof_log[ivl_die_mask] = reg_inner.predict(X_tr[ivl_die_mask])
        assert not np.isnan(reg_train_oof_log).any(), 'inner OOF coverage bug'
        reg_train_y_for_clf = np.clip(np.expm1(reg_train_oof_log), 0.0, None)
    else:
        raise ValueError(f'unknown mode: {mode}')

    X_tr_aug = np.hstack([X_tr, reg_train_y_for_clf.reshape(-1, 1)])

    # ── Stage 2 clf ──
    clf_params = dict(hp)
    clf_params['objective'] = 'binary'
    clf_params['scale_pos_weight'] = clf_spw
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr_aug, y_tr_bin)

    # ── 추론 (모든 X_others 는 reg_full 사용) ──
    results = []
    for X_o in X_others:
        reg_log_o = reg_full.predict(X_o)
        reg_y_o   = np.clip(np.expm1(reg_log_o), 0.0, None)
        X_o_aug   = np.hstack([X_o, reg_y_o.reshape(-1, 1)])
        prob_o = clf.predict_proba(X_o_aug)[:, 1]
        prob_o = np.clip(prob_o, 0.0, 1.0)
        final_o = prob_o * reg_y_o
        results.append((prob_o, reg_y_o, final_o))
    return results


def _mean_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units))
    cnt      = np.zeros(len(unique_units))
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))

print('helpers 정의 완료')

helpers 정의 완료


## 6. Refit 실행 함수 (mode 별)

In [6]:
def run_refit(mode):
    """5-fold refit + die/unit 예측 수집. Returns dict of arrays + metrics."""
    print(f'\n=== Refit (mode={mode}) ===')
    t0 = time.time()

    oof_die_prob   = np.full(n_train_die, np.nan)
    oof_die_reg    = np.full(n_train_die, np.nan)
    oof_die_pred   = np.full(n_train_die, np.nan)
    val_die_prob   = np.zeros(n_val_die)
    val_die_reg    = np.zeros(n_val_die)
    val_die_pred   = np.zeros(n_val_die)
    test_die_prob  = np.zeros(n_test_die)
    test_die_reg   = np.zeros(n_test_die)
    test_die_pred  = np.zeros(n_test_die)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unit_ids_train_unique[tr_uidx]
        vl_units = unit_ids_train_unique[vl_uidx]
        tr_die_mask = np.isin(uid_train_die, tr_units)
        vl_die_mask = np.isin(uid_train_die, vl_units)
        X_tr   = X_train_die[tr_die_mask]
        X_vl   = X_train_die[vl_die_mask]
        y_tr   = y_train_die_broadcast[tr_die_mask]
        yb_tr  = y_bin_die_broadcast[tr_die_mask]
        uid_tr = uid_train_die[tr_die_mask]

        results = _train_path_b(
            X_tr, y_tr, yb_tr,
            [X_vl, X_val_die, X_test_die],
            hp_best, best_w0, best_reg_obj, best_clf_spw,
            mode=mode, uid_tr=uid_tr, k_inner=K_INNER, seed=SEED,
        )
        (p_vl, r_vl, f_vl), (p_v, r_v, f_v), (p_t, r_t, f_t) = results

        oof_die_prob[vl_die_mask] = p_vl
        oof_die_reg[vl_die_mask]  = r_vl
        oof_die_pred[vl_die_mask] = f_vl
        val_die_prob  += p_v / N_FOLDS
        val_die_reg   += r_v / N_FOLDS
        val_die_pred  += f_v / N_FOLDS
        test_die_prob += p_t / N_FOLDS
        test_die_reg  += r_t / N_FOLDS
        test_die_pred += f_t / N_FOLDS

        print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

    assert not np.isnan(oof_die_prob).any()
    assert not np.isnan(oof_die_reg).any()
    assert not np.isnan(oof_die_pred).any()

    # ── unit 집계 ──
    oof_unit_arr,  oof_unit_ids  = _mean_die_to_unit(oof_die_pred,  uid_train_die)
    val_unit_arr,  val_unit_ids  = _mean_die_to_unit(val_die_pred,  uid_val_die)
    test_unit_arr, test_unit_ids = _mean_die_to_unit(test_die_pred, uid_test_die)

    oof_unit_s  = pd.Series(oof_unit_arr,  index=oof_unit_ids).reindex(y_train_unit.index)
    val_unit_s  = pd.Series(val_unit_arr,  index=val_unit_ids).reindex(y_val_unit.index)
    test_unit_s = pd.Series(test_unit_arr, index=test_unit_ids).reindex(y_test_unit.index)

    oof_rmse  = float(np.sqrt(np.mean((oof_unit_s.values  - y_train_unit.values) ** 2)))
    val_rmse  = float(np.sqrt(np.mean((val_unit_s.values  - y_val_unit.values)   ** 2)))
    test_rmse = float(np.sqrt(np.mean((test_unit_s.values - y_test_unit.values)  ** 2)))

    elapsed = time.time() - t0
    print(f'[mode={mode}] elapsed={elapsed:.0f}s  '
          f'oof={oof_rmse:.6f}  val={val_rmse:.6f}  test={test_rmse:.6f}')

    return {
        'mode': mode,
        'elapsed': elapsed,
        'oof_die_prob': oof_die_prob, 'oof_die_reg': oof_die_reg, 'oof_die_pred': oof_die_pred,
        'val_die_prob': val_die_prob, 'val_die_reg': val_die_reg, 'val_die_pred': val_die_pred,
        'test_die_prob': test_die_prob, 'test_die_reg': test_die_reg, 'test_die_pred': test_die_pred,
        'oof_unit_s': oof_unit_s, 'val_unit_s': val_unit_s, 'test_unit_s': test_unit_s,
        'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
    }

print('run_refit 정의 완료')

run_refit 정의 완료


## 7. mode A — in-sample (대조군, 기존 방식 재현)

In [7]:
res_insample = run_refit('insample')


=== Refit (mode=insample) ===
  fold 1/5 done (70s)
  fold 2/5 done (137s)
  fold 3/5 done (208s)
  fold 4/5 done (275s)
  fold 5/5 done (341s)
[mode=insample] elapsed=341s  oof=0.005495  val=0.005709  test=0.008412


## 8. mode B — inner-OOF (실험군, 개선)

In [8]:
res_innerOOF = run_refit('innerOOF')


=== Refit (mode=innerOOF) ===
  fold 1/5 done (212s)
  fold 2/5 done (417s)
  fold 3/5 done (637s)
  fold 4/5 done (857s)
  fold 5/5 done (1008s)
[mode=innerOOF] elapsed=1008s  oof=0.005497  val=0.005709  test=0.008412


## 9. 비교 표 + csv 저장

In [9]:
def _save_mode(res, out_dir):
    """die/unit csv + meta.json 저장."""
    def _build_die_df(uid_arr, die_id_arr, position_arr, prob, reg, pred, y_unit):
        df = pd.DataFrame({
            KEY_COL:     uid_arr,
            DIE_KEY_COL: die_id_arr,
            'position':  position_arr,
            'prob':      prob,
            'reg':       reg,
            'pred':      pred,
        })
        if y_unit is not None:
            df[TARGET_COL] = df[KEY_COL].map(y_unit)
        return df

    _build_die_df(
        uid_train_die, xs_train_die[DIE_KEY_COL].values, xs_train_die['position'].values,
        res['oof_die_prob'], res['oof_die_reg'], res['oof_die_pred'], y_train_unit,
    ).to_csv(os.path.join(out_dir, 'oof_die.csv'), index=False)
    _build_die_df(
        uid_val_die, xs_val_die[DIE_KEY_COL].values, xs_val_die['position'].values,
        res['val_die_prob'], res['val_die_reg'], res['val_die_pred'], y_val_unit,
    ).to_csv(os.path.join(out_dir, 'val_die.csv'), index=False)
    _build_die_df(
        uid_test_die, xs_test_die[DIE_KEY_COL].values, xs_test_die['position'].values,
        res['test_die_prob'], res['test_die_reg'], res['test_die_pred'], y_test_unit,
    ).to_csv(os.path.join(out_dir, 'test_die.csv'), index=False)

    def _build_unit_df(unit_pred_s, y_unit):
        return pd.DataFrame({
            KEY_COL:    unit_pred_s.index.values,
            'pred':     unit_pred_s.values,
            TARGET_COL: y_unit.reindex(unit_pred_s.index).values,
        })

    _build_unit_df(res['oof_unit_s'],  y_train_unit).to_csv(os.path.join(out_dir, 'oof_unit.csv'),  index=False)
    _build_unit_df(res['val_unit_s'],  y_val_unit ).to_csv(os.path.join(out_dir, 'val_unit.csv'),  index=False)
    _build_unit_df(res['test_unit_s'], y_test_unit).to_csv(os.path.join(out_dir, 'test_unit.csv'), index=False)

    meta = {
        'exp_id':            EXP_ID,
        'exp_memo':          EXP_MEMO,
        'mode':              res['mode'],
        'model':             'Reverse Two-Stage (lgbm reg → lgbm clf, weighted MSE, log1p)',
        'path_type':         'B (reverse: reg → clf)',
        'stage2_feat_mode':  res['mode'],
        'k_inner':           (K_INNER if res['mode'] == 'innerOOF' else None),
        'target_transform':  TARGET_TRANSFORM,
        'die_to_unit_agg':   'mean',
        'training_level':    'die-level broadcast',
        'n_folds':           N_FOLDS,
        'oof_rmse':          res['oof_rmse'],
        'val_rmse':          res['val_rmse'],
        'test_rmse':         res['test_rmse'],
        'elapsed_seconds':   res['elapsed'],
        'preprocess_PARAMS': PARAMS,
        'effective_pp_params': pp['effective_params'],
        'best_params_source':  BEST_PARAMS_PATH,
        'best_params':       best_params,
        'CLIP_Y_EXTREME':    CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED':              int(SEED),
    }
    with open(os.path.join(out_dir, 'meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=2, ensure_ascii=False, default=str)

    return meta


meta_insample = _save_mode(res_insample, OUT_DIR['insample'])
meta_innerOOF = _save_mode(res_innerOOF, OUT_DIR['innerOOF'])

for mode, out_dir in OUT_DIR.items():
    print(f'\n[{mode}] saved to {out_dir}')
    for f_ in sorted(os.listdir(out_dir)):
        sz = os.path.getsize(os.path.join(out_dir, f_)) / 1024
        print(f'  {f_:35s}  {sz:>10,.1f} KB')


[insample] saved to c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse_innerOOF\insample
  meta.json                                   2.2 KB
  oof_die.csv                             9,884.6 KB
  oof_unit.csv                              970.7 KB
  test_die.csv                            3,301.4 KB
  test_unit.csv                             323.5 KB
  val_die.csv                             3,301.1 KB
  val_unit.csv                              323.4 KB

[innerOOF] saved to c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse_innerOOF\innerOOF
  meta.json                                   2.2 KB
  oof_die.csv                             9,885.1 KB
  oof_unit.csv                              970.8 KB
  test_die.csv                            3,301.7 KB
  test_unit.csv                             323.5 KB
  val_die.csv                             3,301.1 KB
  val_unit.csv                              323.5 KB


## 10. 비교 요약

두 모드의 OOF / val / test RMSE 와 잔차 상관 비교.

In [10]:
# ── RMSE 표 ──
summary = pd.DataFrame({
    'mode':     ['insample', 'innerOOF'],
    'oof':      [res_insample['oof_rmse'],  res_innerOOF['oof_rmse']],
    'val':      [res_insample['val_rmse'],  res_innerOOF['val_rmse']],
    'test':     [res_insample['test_rmse'], res_innerOOF['test_rmse']],
    'elapsed':  [res_insample['elapsed'],   res_innerOOF['elapsed']],
})
summary['Δ_oof']  = summary['oof']  - summary.loc[0, 'oof']
summary['Δ_val']  = summary['val']  - summary.loc[0, 'val']
summary['Δ_test'] = summary['test'] - summary.loc[0, 'test']

print('=' * 90)
print('  Reverse Two-Stage refit 비교 (in-sample vs inner-OOF)')
print('=' * 90)
print(summary.to_string(index=False, float_format=lambda x: f'{x:.6f}' if abs(x) < 100 else f'{x:.0f}'))
print('-' * 90)
print(f'  ts-reverse-hpo-001 원본 (참고): oof=0.005495, val=0.005709, test=0.008412')
print('  insample 모드는 위 값과 거의 일치해야 정상 (재현 검증).')
print('=' * 90)

# ── unit-level 잔차 상관 ──
for split_name, s_a, s_b, y_ref in [
    ('OOF',  res_insample['oof_unit_s'],  res_innerOOF['oof_unit_s'],  y_train_unit),
    ('val',  res_insample['val_unit_s'],  res_innerOOF['val_unit_s'],  y_val_unit),
    ('test', res_insample['test_unit_s'], res_innerOOF['test_unit_s'], y_test_unit),
]:
    res_a = s_a.values - y_ref.reindex(s_a.index).values
    res_b = s_b.values - y_ref.reindex(s_b.index).values
    pred_corr = float(np.corrcoef(s_a.values, s_b.values)[0, 1])
    res_corr  = float(np.corrcoef(res_a,        res_b       )[0, 1])
    print(f'[{split_name:5s}] pred corr = {pred_corr:.4f} | residual corr = {res_corr:.4f}')

print('\n→ residual corr 가 1.0 미만이면 두 모드의 오차 패턴이 달라 stacking 다양성 확보 가능.')
print('→ val_rmse 가 inner-OOF 에서 낮으면 추가 HPO 가치 있음.')

  Reverse Two-Stage refit 비교 (in-sample vs inner-OOF)
    mode      oof      val     test  elapsed    Δ_oof     Δ_val   Δ_test
insample 0.005495 0.005709 0.008412      341 0.000000  0.000000 0.000000
innerOOF 0.005497 0.005709 0.008412     1008 0.000002 -0.000000 0.000001
------------------------------------------------------------------------------------------
  ts-reverse-hpo-001 원본 (참고): oof=0.005495, val=0.005709, test=0.008412
  insample 모드는 위 값과 거의 일치해야 정상 (재현 검증).
[OOF  ] pred corr = 0.9951 | residual corr = 0.9998
[val  ] pred corr = 0.9990 | residual corr = 1.0000
[test ] pred corr = 0.9990 | residual corr = 1.0000

→ residual corr 가 1.0 미만이면 두 모드의 오차 패턴이 달라 stacking 다양성 확보 가능.
→ val_rmse 가 inner-OOF 에서 낮으면 추가 HPO 가치 있음.


## 11. 결과 요약 (stacking 통합 가이드)

In [11]:
print('=' * 90)
print(f' Reverse Two-Stage refit (in-sample vs inner-OOF) — 결과 요약')
print('=' * 90)
print(f'  EXP_ID            : {EXP_ID}')
print(f'  Best HP source    : {BEST_PARAMS_PATH}')
print(f'  PP source         : 03b log1p preset (ts-reverse 본인 best)')
print(f'  N_FOLDS / K_INNER : {N_FOLDS} / {K_INNER}')
print(f'  feat cols (clean) : {len(feat_cols_clean)}')
print(f'  best_w0           : {best_w0:.4f}')
print(f'  best_reg_obj      : {best_reg_obj}')
print(f'  best_clf_spw      : {best_clf_spw}')
print('-' * 90)
print(f'  {"mode":12s}  {"OOF":>11s}  {"val":>11s}  {"test":>11s}  {"elapsed":>9s}')
for r in [res_insample, res_innerOOF]:
    print(f'  {r["mode"]:12s}  {r["oof_rmse"]:11.6f}  {r["val_rmse"]:11.6f}  {r["test_rmse"]:11.6f}  {r["elapsed"]:9.0f}s')
print('-' * 90)
delta_val  = res_innerOOF['val_rmse']  - res_insample['val_rmse']
delta_test = res_innerOOF['test_rmse'] - res_insample['test_rmse']
print(f'  Δ(innerOOF − insample): val={delta_val:+.6f}, test={delta_test:+.6f}')
if delta_val < -1e-6:
    print('  → inner-OOF 가 val 기준 우세. HPO 재실행으로 추가 개선 여지 있음.')
elif delta_val > 1e-6:
    print('  → in-sample 이 우세. inner-OOF 효과 미미하거나 역효과.')
else:
    print('  → 두 모드 사실상 동등. 잔차 corr 검사로 stacking 다양성만 점검.')
print('-' * 90)
print(f'  Stacking 통합 후보 경로:')
print(f'    {OUT_DIR["insample"]}/oof_unit.csv, val_unit.csv, test_unit.csv')
print(f'    {OUT_DIR["innerOOF"]}/oof_unit.csv, val_unit.csv, test_unit.csv')
print('=' * 90)

 Reverse Two-Stage refit (in-sample vs inner-OOF) — 결과 요약
  EXP_ID            : ts-reverse-innerOOF-001
  Best HP source    : c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\_temp\two_stage_reverse\best_params.json
  PP source         : 03b log1p preset (ts-reverse 본인 best)
  N_FOLDS / K_INNER : 5 / 5
  feat cols (clean) : 568
  best_w0           : 0.1767
  best_reg_obj      : regression
  best_clf_spw      : 2.43
------------------------------------------------------------------------------------------
  mode                  OOF          val         test    elapsed
  insample         0.005495     0.005709     0.008412        341s
  innerOOF         0.005497     0.005709     0.008412       1008s
------------------------------------------------------------------------------------------
  Δ(innerOOF − insample): val=-0.000000, test=+0.000001
  → 두 모드 사실상 동등. 잔차 corr 검사로 stacking 다양성만 점검.
------------------------------------------------------------------------------------------
  Stacking 통합